# CLOAK Campaign Simulator

Drag the sliders to explore how different audience sizes, ad budgets, and campaign settings affect your probability of funding.

**Run this single cell, then use the sliders below.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from dataclasses import replace

from src.config import CloakConfig, CampaignConfig, FeeStructure
from src.simulation import SimulationInputs, run_simulation
from src.demand import demand_confidence_score
from src.market import saturation_check

sns.set_theme(style='whitegrid', palette='deep')

# -- Fixed product config --
cloak = CloakConfig()
fees = FeeStructure()
SEED = 42
N_RUNS = 1_000  # fast enough for interactive use

# -- Sliders --
style = {'description_width': '140px'}
slider_layout = widgets.Layout(width='420px')

email_slider = widgets.IntSlider(
    value=50, min=0, max=10_000, step=50,
    description='Email list:', style=style, layout=slider_layout,
    continuous_update=False)
ig_slider = widgets.IntSlider(
    value=124, min=0, max=10_000, step=50,
    description='IG followers:', style=style, layout=slider_layout,
    continuous_update=False)
fb_slider = widgets.IntSlider(
    value=69, min=0, max=10_000, step=50,
    description='FB followers:', style=style, layout=slider_layout,
    continuous_update=False)
ad_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=500.0, step=5.0,
    description='Daily ad budget ($):', style=style, layout=slider_layout,
    continuous_update=False)
pr_slider = widgets.IntSlider(
    value=0, min=0, max=10, step=1,
    description='PR/media hits:', style=style, layout=slider_layout,
    continuous_update=False)
site_slider = widgets.IntSlider(
    value=100, min=0, max=10_000, step=50,
    description='Monthly visitors:', style=style, layout=slider_layout,
    continuous_update=False)
goal_slider = widgets.IntSlider(
    value=15_000, min=5_000, max=100_000, step=1_000,
    description='Funding goal ($):', style=style, layout=slider_layout,
    continuous_update=False)
duration_slider = widgets.IntSlider(
    value=30, min=15, max=60, step=1,
    description='Campaign days:', style=style, layout=slider_layout,
    continuous_update=False)
eb_price_slider = widgets.FloatSlider(
    value=149.99, min=99.0, max=179.99, step=5.0,
    description='Early bird price ($):', style=style, layout=slider_layout,
    continuous_update=False)
eb_qty_slider = widgets.IntSlider(
    value=50, min=0, max=500, step=10,
    description='Early bird qty:', style=style, layout=slider_layout,
    continuous_update=False)

# -- Output area --
output = widgets.Output(layout=widgets.Layout(width='100%'))


def run_and_display(**kwargs):
    with output:
        clear_output(wait=True)

        campaign = CampaignConfig(goal=kwargs['goal'], duration_days=kwargs['duration'])
        this_cloak = replace(cloak, early_bird_price=kwargs['eb_price'], early_bird_quantity=kwargs['eb_qty'])

        inputs = SimulationInputs(
            email_list=kwargs['email'],
            ig_followers=kwargs['ig'],
            fb_followers=kwargs['fb'],
            daily_ad_budget=kwargs['ad_budget'],
            pr_hits=kwargs['pr'],
            monthly_site_visitors=kwargs['site'],
            cloak=this_cloak, campaign=campaign, fees=fees,
        )

        results = run_simulation(inputs, n_runs=N_RUNS, seed=SEED)
        prob = results.probability_of_funding()
        p10, p50, p90 = results.percentiles([10, 50, 90])
        b10, b50, b90 = np.percentile(results.total_backers, [10, 50, 90])
        median_net = float(np.median(results.net_revenue))

        # -- Color coding --
        if prob >= 0.70:
            prob_color, prob_label = '#27ae60', 'STRONG'
        elif prob >= 0.50:
            prob_color, prob_label = '#2980b9', 'VIABLE'
        elif prob >= 0.20:
            prob_color, prob_label = '#f39c12', 'RISKY'
        else:
            prob_color, prob_label = '#e74c3c', 'DO NOT LAUNCH'

        # -- Scorecard HTML --
        scorecard_html = f"""
        <div style="font-family: monospace; background: #1a1a2e; color: #eee;
                    padding: 20px; border-radius: 10px; margin-bottom: 15px;">
          <div style="text-align: center; margin-bottom: 12px;">
            <span style="font-size: 48px; font-weight: bold; color: {prob_color};">
              {prob:.0%}
            </span>
            <br/>
            <span style="font-size: 16px; color: {prob_color};">{prob_label}</span>
            <span style="font-size: 14px; color: #888;"> -- probability of funding</span>
          </div>
          <table style="width: 100%; color: #ccc; font-size: 14px; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #333;">
              <td style="padding: 4px 8px;">Funding goal</td>
              <td style="text-align: right; padding: 4px 8px;">${campaign.goal:,.0f}</td>
            </tr>
            <tr style="border-bottom: 1px solid #333;">
              <td style="padding: 4px 8px;">Expected raised (10th / 50th / 90th)</td>
              <td style="text-align: right; padding: 4px 8px;">
                ${p10:,.0f} / ${p50:,.0f} / ${p90:,.0f}
              </td>
            </tr>
            <tr style="border-bottom: 1px solid #333;">
              <td style="padding: 4px 8px;">Expected backers (10th / 50th / 90th)</td>
              <td style="text-align: right; padding: 4px 8px;">
                {b10:.0f} / {b50:.0f} / {b90:.0f}
              </td>
            </tr>
            <tr>
              <td style="padding: 4px 8px;">Net revenue (median)</td>
              <td style="text-align: right; padding: 4px 8px; color: {'#27ae60' if median_net > 0 else '#e74c3c'};">
                ${median_net:,.0f}
              </td>
            </tr>
          </table>
        </div>
        """
        display(HTML(scorecard_html))

        # -- Fan chart --
        fig, ax = plt.subplots(figsize=(11, 4.5))
        days = np.arange(1, campaign.duration_days + 1)
        t10 = np.percentile(results.daily_trajectories, 10, axis=0)
        t25 = np.percentile(results.daily_trajectories, 25, axis=0)
        t50 = np.percentile(results.daily_trajectories, 50, axis=0)
        t75 = np.percentile(results.daily_trajectories, 75, axis=0)
        t90 = np.percentile(results.daily_trajectories, 90, axis=0)

        ax.fill_between(days, t10, t90, alpha=0.15, color='steelblue', label='10th-90th')
        ax.fill_between(days, t25, t75, alpha=0.3, color='steelblue', label='25th-75th')
        ax.plot(days, t50, color='steelblue', linewidth=2, label='Median')
        ax.axhline(y=campaign.goal, color='red', linestyle='--', linewidth=1.5,
                   label=f'Goal: ${campaign.goal:,}')
        ax.set_xlabel('Campaign Day')
        ax.set_ylabel('Cumulative Raised ($)')
        ax.set_title('Funding Trajectory')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()
        plt.show()

        # -- Revenue distribution histogram --
        fig2, ax2 = plt.subplots(figsize=(11, 3))
        ax2.hist(results.total_raised, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
        ax2.axvline(x=campaign.goal, color='red', linestyle='--', linewidth=1.5, label=f'Goal: ${campaign.goal:,}')
        ax2.axvline(x=p50, color='#27ae60', linestyle='-', linewidth=1.5, label=f'Median: ${p50:,.0f}')
        ax2.set_xlabel('Total Raised ($)')
        ax2.set_ylabel('Simulation Runs')
        ax2.set_title(f'Distribution of Outcomes ({N_RUNS:,} runs)')
        ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax2.legend(fontsize=9)
        plt.tight_layout()
        plt.show()


# -- Wire up interactive output --
interactive_out = widgets.interactive_output(run_and_display, {
    'email': email_slider,
    'ig': ig_slider,
    'fb': fb_slider,
    'ad_budget': ad_slider,
    'pr': pr_slider,
    'site': site_slider,
    'goal': goal_slider,
    'duration': duration_slider,
    'eb_price': eb_price_slider,
    'eb_qty': eb_qty_slider,
})

# -- Layout --
audience_box = widgets.VBox([
    widgets.HTML('<h3 style="margin: 0 0 4px 0;">Audience</h3>'),
    email_slider, ig_slider, fb_slider, ad_slider, pr_slider, site_slider,
], layout=widgets.Layout(padding='10px'))

campaign_box = widgets.VBox([
    widgets.HTML('<h3 style="margin: 0 0 4px 0;">Campaign</h3>'),
    goal_slider, duration_slider, eb_price_slider, eb_qty_slider,
], layout=widgets.Layout(padding='10px'))

controls = widgets.VBox([audience_box, campaign_box],
    layout=widgets.Layout(width='480px', border='1px solid #ddd',
                          border_radius='8px', padding='10px'))

results_panel = widgets.VBox([interactive_out],
    layout=widgets.Layout(flex='1', padding='0 0 0 15px'))

app = widgets.HBox([controls, results_panel],
    layout=widgets.Layout(width='100%', align_items='flex-start'))

display(app)